## Open the project folders

In [1]:
from pathlib import Path
import csv, json, hashlib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "dataset_sources.json").is_file():
    raise FileNotFoundError("Open this notebook from the extracted FallDetection_Step2 folder in JupyterLab, then restart the kernel and try again.")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
manifest = json.loads((ROOT / "dataset_sources.json").read_text())
print("Project folder:", ROOT)
print("Dataset:", manifest["dataset"])
print("Expected recordings:", len(manifest["files"]))

Project folder: C:\Users\janit\Desktop\Fall Detection\FallDetection_Step2
Dataset: UP-Fall
Expected recordings: 66


## Verify that the originals are complete

In [2]:
assert len(manifest["files"]) == 66
for info in manifest["files"]:
    path = RAW / info["filename"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path.name}; extract the entire ZIP, not only the notebook.")
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == info["sha256"], f"Original file changed: {path.name}"
print("Verified: all 66 original recordings are present and unchanged.")
print("Subjects: 1, 2, 3, 4, 5, 6")
print("Activities per subject: 11; repetition: Trial 1")

Verified: all 66 original recordings are present and unchanged.
Subjects: 1, 2, 3, 4, 5, 6
Activities per subject: 11; repetition: Trial 1


## Read the belt sensor and inspect data quality

In [3]:
ACTIVITIES = {1:"Forward fall using hands", 2:"Forward fall using knees", 3:"Backward fall",
              4:"Sideways fall", 5:"Fall while attempting to sit", 6:"Walking", 7:"Standing",
              8:"Sitting", 9:"Picking up an object", 10:"Jumping", 11:"Lying"}
SENSOR_COLUMNS = ["acc_x_g", "acc_y_g", "acc_z_g", "gyro_x_deg_s", "gyro_y_deg_s", "gyro_z_deg_s"]

def read_belt(info):
    with (RAW / info["filename"]).open(newline="", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))
    header, units, samples = rows[0], rows[1], rows[2:]
    assert header[15] == "BeltAccelerometer" and header[18] == "BeltAngularVelocity"
    assert header[43:46] == ["Subject", "Activity", "Trial"]
    assert all("(g)" in value for value in units[15:18])
    assert all("(deg/s)" in value for value in units[18:21])
    assert all(len(row) == 47 for row in samples), info["filename"]
    assert len(samples) == info["sample_count"], info["filename"]
    table = pd.DataFrame(samples).iloc[:, [0,15,16,17,18,19,20,43,44,45,46]].copy()
    table.columns = ["timestamp", *SENSOR_COLUMNS, "subject_id", "trial_activity_id", "trial_id", "frame_activity_id"]
    timestamp = pd.to_datetime(table["timestamp"], errors="coerce")
    table["timestamp"] = timestamp
    for col in table.columns[1:]:
        table[col] = pd.to_numeric(table[col], errors="coerce")
    assert table["subject_id"].eq(info["subject_id"]).all(), info["filename"]
    assert table["trial_activity_id"].eq(info["trial_activity_id"]).all(), info["filename"]
    assert table["trial_id"].eq(1).all()
    table.insert(1, "time_s", (timestamp-timestamp.iloc[0]).dt.total_seconds())
    table["frame_activity"] = table["frame_activity_id"].map(ACTIVITIES).fillna("Unmapped")
    table["recording_id"] = Path(info["filename"]).stem
    table["split"] = info["split"]
    invalid_sensor = ~np.isfinite(table[SENSOR_COLUMNS].to_numpy(dtype=float)).all(axis=1)
    valid_label = table["frame_activity_id"].isin(ACTIVITIES)
    table["usable_sensor_and_label"] = ~invalid_sensor & valid_label & timestamp.notna()
    gaps = table["time_s"].diff()
    positive = gaps[gaps > 0]
    report = {"filename":info["filename"], "subject_id":info["subject_id"],
              "trial_activity_id":info["trial_activity_id"], "split":info["split"], "rows":len(table),
              "duration_s":table["time_s"].iloc[-1], "invalid_sensor_rows":int(invalid_sensor.sum()),
              "unmapped_label_rows":int((~valid_label).sum()), "invalid_timestamp_rows":int(timestamp.isna().sum()),
              "nonpositive_time_steps":int(gaps.le(0).sum()), "median_interval_s":positive.median(),
              "largest_gap_s":positive.max(), "gaps_over_0_1_s":int(gaps.gt(0.1).sum())}
    return table, report

recordings = {}
audit_rows = []
for info in manifest["files"]:
    frame, report = read_belt(info)
    recordings[Path(info["filename"]).stem] = frame
    audit_rows.append(report)
audit = pd.DataFrame(audit_rows).sort_values(["subject_id", "trial_activity_id"]).reset_index(drop=True)
print("Recordings loaded:", len(recordings))
print("Total synchronized rows:", int(audit["rows"].sum()))
print("Invalid belt-sensor rows:", int(audit["invalid_sensor_rows"].sum()))
print("Rows with unmapped activity tags:", int(audit["unmapped_label_rows"].sum()))
print("Invalid timestamp rows:", int(audit["invalid_timestamp_rows"].sum()))
print("Nonpositive timestamp steps:", int(audit["nonpositive_time_steps"].sum()))
print("Gaps larger than 0.1 s:", int(audit["gaps_over_0_1_s"].sum()))
print("No rows have been deleted or imputed.")

Recordings loaded: 66
Total synchronized rows: 34535
Invalid belt-sensor rows: 0
Rows with unmapped activity tags: 107
Invalid timestamp rows: 0
Nonpositive timestamp steps: 0
Gaps larger than 0.1 s: 1185
No rows have been deleted or imputed.


## Keep different participants in different splits

| Split | Subjects | Later purpose |
|---|---|---|
| Training | 1, 2, 3 | Learn model parameters |
| Validation | 4 | Choose settings and thresholds |
| Test | 5, 6 | Final check after choices are frozen |


In [4]:
split_subjects = {"train": {1,2,3}, "validation": {4}, "test": {5,6}}
assert split_subjects["train"].isdisjoint(split_subjects["validation"])
assert split_subjects["train"].isdisjoint(split_subjects["test"])
assert split_subjects["validation"].isdisjoint(split_subjects["test"])
for row in audit.itertuples():
    assert row.subject_id in split_subjects[row.split]
for subject, part in audit.groupby("subject_id"):
    assert len(part) == 11 and set(part["trial_activity_id"]) == set(range(1,12))
print("Participant split checked: no participant appears in more than one split.")
for split in ["train", "validation", "test"]:
    part = audit[audit["split"].eq(split)]
    print(f"{split:10s} | subjects {sorted(split_subjects[split])} | {len(part):2d} recordings | {part['rows'].sum():,} rows")

Participant split checked: no participant appears in more than one split.
train      | subjects [1, 2, 3] | 33 recordings | 17,679 rows
validation | subjects [4] | 11 recordings | 5,938 rows
test       | subjects [5, 6] | 22 recordings | 10,918 rows


## Save the prepared data and send the summary
This writes one extracted belt CSV per original recording and two reports. The originals stay unchanged. Missing or invalid readings, if present, remain flagged rather than silently repaired.

Send me this cell's output, including any quality flags. We will set the time-window and gap-handling rules before training.

In [5]:
for recording_id, frame in recordings.items():
    target = PROCESSED / frame["split"].iloc[0]
    target.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target / f"{recording_id}_belt.csv", index=False)
audit.to_csv(RESULTS / "data_quality_report.csv", index=False)
split_manifest = audit[["filename", "subject_id", "trial_activity_id", "split", "rows"]]
split_manifest.to_csv(RESULTS / "participant_split.csv", index=False)
assert len(list(PROCESSED.glob("*/*_belt.csv"))) == 66
print("Dataset preparation complete.")
print("Original recordings: 66")
print("Extracted belt recordings: 66")
print("Participants: 6; activity types: 11; trial: 1")
print("Training: 33 recordings; validation: 11; test: 22")
print("Reports saved in:", RESULTS)
print("Model training has not started yet.")

Dataset preparation complete.
Original recordings: 66
Extracted belt recordings: 66
Participants: 6; activity types: 11; trial: 1
Training: 33 recordings; validation: 11; test: 22
Reports saved in: C:\Users\janit\Desktop\Fall Detection\FallDetection_Step2\results
Model training has not started yet.
